In [ ]:
import torch
import hashlib
import json
from torch.utils.data import Subset

NUM_CLASSES = 43

# Use full dataset if available, otherwise use normal dataset
raw_train_dataset = train_dataset_full if "train_dataset_full" in globals() else train_dataset
raw_test_dataset = test_dataset_full if "test_dataset_full" in globals() else test_dataset

def get_image_hash(img):
    return hashlib.md5(img.cpu().numpy().tobytes()).hexdigest()

def clean_dataset(dataset, name):
    valid_indices = []
    removed = []
    seen_images = set()

    for i in range(len(dataset)):
        try:
            img, label = dataset[i]

            if not isinstance(label, int):
                removed.append((i, label, "Invalid label type"))
                continue

            if label < 0 or label >= NUM_CLASSES:
                removed.append((i, label, "Label out of range"))
                continue

            if not torch.is_tensor(img):
                removed.append((i, label, "Image is not tensor"))
                continue

            if len(img.shape) != 3:
                removed.append((i, label, "Invalid image shape"))
                continue

            if img.shape[0] not in [1, 3]:
                removed.append((i, label, "Invalid channel count"))
                continue

            if torch.isnan(img).any() or torch.isinf(img).any():
                removed.append((i, label, "Corrupted image"))
                continue

            if torch.max(img) == torch.min(img):
                removed.append((i, label, "Blank image"))
                continue

            img_hash = get_image_hash(img)

            if img_hash in seen_images:
                removed.append((i, label, "Duplicate image"))
                continue

            seen_images.add(img_hash)
            valid_indices.append(i)

        except Exception as e:
            removed.append((i, "unknown", str(e)))

    cleaned_dataset = Subset(dataset, valid_indices)

    print("\n" + name + " Cleaning Report")
    print("-" * 40)
    print("Original images :", len(dataset))
    print("Clean images    :", len(cleaned_dataset))
    print("Removed images  :", len(removed))

    if removed:
        print("\nFirst removed samples:")
        for item in removed[:10]:
            print(item)
    else:
        print("No invalid images found.")

    return cleaned_dataset, removed

clean_train_dataset, train_removed = clean_dataset(raw_train_dataset, "Training Dataset")
clean_test_dataset, test_removed = clean_dataset(raw_test_dataset, "Testing Dataset")

cleaning_report = {
    "train_original": len(raw_train_dataset),
    "train_cleaned": len(clean_train_dataset),
    "train_removed": len(train_removed),
    "test_original": len(raw_test_dataset),
    "test_cleaned": len(clean_test_dataset),
    "test_removed": len(test_removed)
}

with open("dataset_cleaning_report.json", "w") as f:
    json.dump(cleaning_report, f, indent=4)

print("\nDataset cleaning completed successfully.")
print("Cleaning report saved as dataset_cleaning_report.json")

# Use these cleaned datasets for next steps
train_dataset = clean_train_dataset
test_dataset = clean_test_dataset